In [ ]:
# ============================================================
# A0049000 FIRM 확정 기준 firm_quantity 4주 예측 v7
# 개선 방향:
# - 기존 FIRM 확정 기준 split 유지
# - 외부변수 lag 1~16 유지
# - forecast_quantity를 핵심 feature로 추가
# - MA 단독이 아니라 forecast→firm 전환모델 후보 추가
# - calibration 구간에서 RMSE/WAPE 기준으로 후보 선택
# - holdout 성능과 미래 4주 예측 저장
# ============================================================

import warnings
warnings.filterwarnings('ignore')

import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV, HuberRegressor, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DEMAND_PATH = Path('/content/A0049000_최신데이터.xlsx')
EXTERNAL_PATH = Path('/content/외부변수 통합 데이터들.xlsx')
TARGET_PN = 'A0049000'
TARGET_COL = 'Quantity'
FIRM_VALUE = 'FIRM'
FORECAST_VALUE = 'FORECAST'
DATE_COL_EXTERNAL = 'Date'

FORECAST_WEEKS = 4
MAX_EXTERNAL_LAG = 16
HOLDOUT_CONFIRMED_WEEKS = 8
INNER_CAL_CONFIRMED_WEEKS = 4
TOP_EXTERNAL_N = 10
TARGET_RMSE_MAX = 1.0
TARGET_WAPE_MAX = 0.10
OUTPUT_PREFIX = f'{TARGET_PN}_firm_confirmed_v7_forecast_fix'

# True면 holdout 성능표에서 후보별로 scale/offset을 재탐색하지 않음.
# 실제 운영/발표에서 데이터 누수 방지하려면 True 유지.
STRICT_NO_TEST_LEAK = True

MONTH_MAP = {
    'JANUARY':1,'JAN':1,'1월':1,'01월':1,
    'FEBRUARY':2,'FEB':2,'2월':2,'02월':2,
    'MARCH':3,'MAR':3,'3월':3,'03월':3,
    'APRIL':4,'APR':4,'4월':4,'04월':4,
    'MAY':5,'5월':5,'05월':5,
    'JUNE':6,'JUN':6,'6월':6,'06월':6,
    'JULY':7,'JUL':7,'7월':7,'07월':7,
    'AUGUST':8,'AUG':8,'8월':8,'08월':8,
    'SEPTEMBER':9,'SEP':9,'SEPT':9,'9월':9,'09월':9,
    'OCTOBER':10,'OCT':10,'10월':10,
    'NOVEMBER':11,'NOV':11,'11월':11,
    'DECEMBER':12,'DEC':12,'12월':12,
}


def ensure_files_exist():
    missing = [p for p in [DEMAND_PATH, EXTERNAL_PATH] if not p.exists()]
    if missing and IN_COLAB:
        print('아래 파일 2개를 업로드하세요:')
        print('1) A0049000_최신데이터.xlsx')
        print('2) 외부변수 통합 데이터들.xlsx')
        files.upload()
    missing = [p for p in [DEMAND_PATH, EXTERNAL_PATH] if not p.exists()]
    if missing:
        raise FileNotFoundError('필수 파일 없음: ' + ', '.join(str(p) for p in missing))


def clean_col(c):
    return str(c).replace('\n',' ').replace('\r',' ').strip()


def norm(s):
    return re.sub(r'[^a-z0-9가-힣]+', '', str(s).lower())


def find_col(cols, names, contains=None):
    nmap = {norm(c): c for c in cols}
    for name in names:
        if norm(name) in nmap:
            return nmap[norm(name)]
    if contains:
        for c in cols:
            if any(norm(x) in norm(c) for x in contains):
                return c
    return None


def parse_num(s):
    if pd.isna(s):
        return np.nan
    if isinstance(s, (int, float, np.integer, np.floating)):
        return float(s)
    t = str(s).strip().replace(',', '')
    # 1.139.136 같은 유럽식 천 단위 방어
    if t.count('.') >= 2:
        t = t.replace('.', '')
    return pd.to_numeric(t, errors='coerce')


def make_date(df):
    m_raw = df['Month'].astype(str).str.strip()
    m = m_raw.str.upper().map(MONTH_MAP)
    m = m.fillna(pd.to_numeric(m_raw, errors='coerce'))
    return pd.to_datetime({
        'year': pd.to_numeric(df['Year'], errors='coerce'),
        'month': m,
        'day': pd.to_numeric(df['Day'], errors='coerce')
    }, errors='coerce')


def week_start_monday(s):
    s = pd.to_datetime(s)
    return s - pd.to_timedelta(s.dt.weekday, unit='D')


def rmse(y, p):
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    return float(np.sqrt(np.mean((y-p)**2)))


def wape(y, p):
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    den = np.sum(np.abs(y))
    return np.nan if den == 0 else float(np.sum(np.abs(y-p)) / den)


def mase(y, p, train_y):
    train_y = np.asarray(train_y, dtype=float)
    if len(train_y) < 2:
        return np.nan
    naive = np.mean(np.abs(train_y[1:] - train_y[:-1]))
    return np.nan if naive == 0 else float(np.mean(np.abs(np.asarray(y)-np.asarray(p))) / naive)


def sum_gap_rate(y, p):
    ysum = np.sum(y)
    return np.nan if ysum == 0 else float((np.sum(p)-ysum)/ysum)


def pack_round(x, unit=0.896):
    arr = np.asarray(x, dtype=float)
    arr = np.maximum(arr, 0)
    return np.round(arr / unit) * unit


def eval_row(name, y, p, train_y):
    p = np.maximum(np.asarray(p, dtype=float), 0)
    return {
        'Model': name,
        'Row_N': len(y),
        'RMSE': rmse(y, p),
        'RMSE_PASS': rmse(y, p) <= TARGET_RMSE_MAX,
        'WAPE': wape(y, p),
        'WAPE_PASS': wape(y, p) <= TARGET_WAPE_MAX,
        'MAE': float(mean_absolute_error(y, p)),
        'MASE': mase(y, p, train_y),
        'Sum Gap Rate': sum_gap_rate(y, p),
        'Actual Sum': float(np.sum(y)),
        'Pred Sum': float(np.sum(p)),
    }


def score_for_select(y, p, train_y):
    r = rmse(y, p)
    w = wape(y, p)
    g = abs(sum_gap_rate(y, p))
    # RMSE<=1 후보가 있으면 유리, 없으면 RMSE와 WAPE를 동시에 낮춤
    penalty = max(0, r - TARGET_RMSE_MAX) * 2.0 + max(0, w - TARGET_WAPE_MAX) * 1.5
    return 0.55*r + 0.35*w + 0.10*g + penalty


def best_scale_offset(y, raw_pred, allow_offset=True):
    # calibration에서만 사용. 과적합 방지 위해 scale/offset 범위를 제한.
    y = np.asarray(y, dtype=float)
    raw_pred = np.asarray(raw_pred, dtype=float)
    best = (np.inf, 1.0, 0.0, np.maximum(raw_pred, 0))
    scales = np.round(np.arange(0.55, 1.61, 0.01), 2)
    offsets = np.arange(-2.0, 2.01, 0.25) if allow_offset else [0.0]
    for s in scales:
        for b in offsets:
            p = pack_round(np.maximum(raw_pred * s + b, 0))
            sc = score_for_select(y, p, y)
            if sc < best[0]:
                best = (sc, float(s), float(b), p)
    return best[1], best[2]


def load_demand_weekly(path):
    raw = pd.read_excel(path)
    raw.columns = [clean_col(c) for c in raw.columns]
    if 'ZF_PN' in raw.columns:
        raw = raw[raw['ZF_PN'].astype(str).str.strip().eq(TARGET_PN)].copy()
    firm_col = find_col(raw.columns, ['Firm/Forecast', 'Firm Forecast'], contains=['firm','forecast'])
    if firm_col is None:
        raise ValueError('Firm/Forecast 컬럼을 찾지 못했습니다.')
    raw['date'] = make_date(raw)
    raw[TARGET_COL] = pd.to_numeric(raw[TARGET_COL], errors='coerce')
    raw['CUM_QTY_NUM'] = raw['CUM_QTY'].apply(parse_num) if 'CUM_QTY' in raw.columns else np.nan
    raw = raw.dropna(subset=['date', TARGET_COL]).copy()
    raw['week'] = week_start_monday(raw['date'])
    raw[firm_col] = raw[firm_col].astype(str).str.upper().str.strip()

    aggs = []
    for status in [FIRM_VALUE, FORECAST_VALUE]:
        part = raw[raw[firm_col].eq(status)].copy()
        g = part.groupby('week').agg(
            qty=(TARGET_COL, 'sum'),
            cnt=(TARGET_COL, 'size'),
            mean=(TARGET_COL, 'mean'),
            max=(TARGET_COL, 'max'),
            std=(TARGET_COL, 'std'),
            order_n=('Order', 'nunique') if 'Order' in part.columns else (TARGET_COL, 'size'),
            cum_max=('CUM_QTY_NUM', 'max'),
        )
        g = g.add_prefix(status.lower() + '_')
        aggs.append(g)

    weekly = pd.concat(aggs, axis=1).reset_index()
    full = pd.DataFrame({'week': pd.date_range(weekly['week'].min(), weekly['week'].max(), freq='W-MON')})
    weekly = full.merge(weekly, on='week', how='left').fillna(0)
    weekly = weekly.rename(columns={'firm_qty':'firm_quantity', 'forecast_qty':'forecast_quantity'})

    confirmed_weeks = weekly.loc[weekly['firm_quantity'] > 0, 'week'].copy()
    return weekly, raw, firm_col, confirmed_weeks


def load_external_weekly(path):
    ext = pd.read_excel(path)
    ext.columns = [clean_col(c) for c in ext.columns]
    date_col = find_col(ext.columns, [DATE_COL_EXTERNAL, 'Date', '날짜'], contains=['date','날짜'])
    if date_col is None:
        raise ValueError('외부변수 Date 컬럼을 찾지 못했습니다.')
    ext[date_col] = pd.to_datetime(ext[date_col], errors='coerce')
    ext = ext.dropna(subset=[date_col]).copy()
    ext['week'] = week_start_monday(ext[date_col])
    num_cols = [c for c in ext.columns if c not in [date_col, 'week']]
    for c in num_cols:
        ext[c] = pd.to_numeric(ext[c], errors='coerce')
    num_cols = [c for c in num_cols if pd.api.types.is_numeric_dtype(ext[c])]
    weekly_ext = ext.groupby('week', as_index=False)[num_cols].mean().sort_values('week')
    return weekly_ext, num_cols


def make_base(weekly, weekly_ext):
    last_confirmed = weekly.loc[weekly['firm_quantity'] > 0, 'week'].max()
    future_weeks = list(pd.date_range(last_confirmed + pd.Timedelta(days=7), periods=FORECAST_WEEKS, freq='W-MON'))
    max_week = max(weekly['week'].max(), future_weeks[-1])
    base = pd.DataFrame({'week': pd.date_range(weekly['week'].min(), max_week, freq='W-MON')})
    df = base.merge(weekly, on='week', how='left')
    df = df.merge(weekly_ext, on='week', how='left')
    fill_cols = [c for c in df.columns if c != 'week']
    df[fill_cols] = df[fill_cols].ffill().bfill().fillna(0)
    return df, future_weeks


def engineer_features(df, ext_cols):
    df = df.copy().sort_values('week').reset_index(drop=True)
    df['weekofyear'] = df['week'].dt.isocalendar().week.astype(int)
    df['month_num'] = df['week'].dt.month

    # firm 자기 이력
    for lag in range(1, 17):
        df[f'firm_lag{lag}'] = df['firm_quantity'].shift(lag)
        df[f'forecast_lag{lag}'] = df['forecast_quantity'].shift(lag)
        df[f'forecast_cnt_lag{lag}'] = df['forecast_cnt'].shift(lag) if 'forecast_cnt' in df.columns else 0
        df[f'forecast_mean_lag{lag}'] = df['forecast_mean'].shift(lag) if 'forecast_mean' in df.columns else 0
        df[f'forecast_max_lag{lag}'] = df['forecast_max'].shift(lag) if 'forecast_max' in df.columns else 0

    for win in [2,3,4,6,8,12,16]:
        df[f'firm_ma{win}'] = df['firm_quantity'].shift(1).rolling(win).mean()
        df[f'firm_med{win}'] = df['firm_quantity'].shift(1).rolling(win).median()
        df[f'forecast_ma{win}'] = df['forecast_quantity'].shift(1).rolling(win).mean()
        df[f'forecast_med{win}'] = df['forecast_quantity'].shift(1).rolling(win).median()
        ratio = df['firm_quantity'].shift(1) / df['forecast_quantity'].shift(1).replace(0, np.nan)
        df[f'ratio_med{win}'] = ratio.rolling(win).median()
        df[f'ratio_mean{win}'] = ratio.rolling(win).mean()
        df[f'ratio_pred_med{win}'] = df['forecast_quantity'] * df[f'ratio_med{win}']
        df[f'ratio_pred_mean{win}'] = df['forecast_quantity'] * df[f'ratio_mean{win}']

    df['firm_trend_1_2'] = df['firm_lag1'] - df['firm_lag2']
    df['firm_trend_4'] = df['firm_lag1'] - df['firm_lag4']
    df['forecast_chg1'] = df['forecast_quantity'] - df['forecast_lag1']
    df['forecast_pct_chg1'] = df['forecast_chg1'] / df['forecast_lag1'].replace(0, np.nan)
    df['forecast_current'] = df['forecast_quantity']
    df['forecast_to_ma4'] = df['forecast_quantity'] / df['forecast_ma4'].replace(0, np.nan)

    # 외부변수 lag 1~16
    for c in ext_cols:
        for lag in range(1, MAX_EXTERNAL_LAG + 1):
            df[f'EXT_L{lag}__{c}'] = df[c].shift(lag)
        df[f'EXT_MA4__{c}'] = df[c].shift(1).rolling(4).mean()
        df[f'EXT_CHG1__{c}'] = df[c].shift(1).pct_change(1)

    return df.replace([np.inf, -np.inf], np.nan)


def simple_ma_candidates(df):
    out = pd.DataFrame({'week': df['week']})
    out['fixed_ma'] = 0.50*df['firm_lag1'] + 0.30*df['firm_ma4'] + 0.20*df['firm_ma8']
    out['recent_ma'] = 0.70*df['firm_lag1'] + 0.30*df['firm_ma4']
    out['conservative'] = 0.60*df['firm_lag1'] + 0.20*df['firm_ma4'] + 0.20*df['firm_med12']
    for w in [2,3,4,6,8,12,16]:
        out[f'ratio_med{w}'] = df[f'ratio_pred_med{w}']
        out[f'ratio_mean{w}'] = df[f'ratio_pred_mean{w}']
    out = out.drop(columns=['week']).clip(lower=0)
    return out


def select_external_features(fit_df, feature_cols):
    # residual 보정용 외부변수는 너무 많이 쓰지 않음
    candidates = [c for c in feature_cols if c.startswith('EXT_')]
    if not candidates:
        return []
    temp = fit_df[candidates + ['residual']].replace([np.inf, -np.inf], np.nan).dropna()
    scores = []
    for c in candidates:
        if temp[c].nunique(dropna=True) < 2:
            continue
        corr = temp[[c, 'residual']].corr().iloc[0,1]
        if pd.notna(corr):
            scores.append((c, abs(corr)))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return [x[0] for x in scores[:TOP_EXTERNAL_N]]


def fit_ml_models(train_df, feature_cols):
    X = train_df[feature_cols]
    y = train_df['firm_quantity']
    models = {
        'ml_ridge': Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler()), ('m', RidgeCV(alphas=[0.01,0.1,1,3,10,30,100,300]))]),
        'ml_huber': Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler()), ('m', HuberRegressor(alpha=0.001, epsilon=1.35, max_iter=1000))]),
        'ml_elastic': Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()), ('m', ElasticNetCV(l1_ratio=[.1,.3,.5,.7,.9], alphas=[.001,.01,.1,1,10], cv=3, max_iter=20000, random_state=42))]),
        'ml_rf': Pipeline([('imp', SimpleImputer(strategy='median')), ('m', RandomForestRegressor(n_estimators=700, max_depth=4, min_samples_leaf=2, random_state=42))]),
        'ml_extratrees': Pipeline([('imp', SimpleImputer(strategy='median')), ('m', ExtraTreesRegressor(n_estimators=700, max_depth=4, min_samples_leaf=2, random_state=43))]),
        'ml_gbr': Pipeline([('imp', SimpleImputer(strategy='median')), ('m', GradientBoostingRegressor(n_estimators=120, max_depth=1, learning_rate=0.05, random_state=44))]),
        'ml_knn5': Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()), ('m', KNeighborsRegressor(n_neighbors=5, weights='distance'))]),
    }
    fitted = {}
    for name, model in models.items():
        try:
            model.fit(X, y)
            fitted[name] = model
        except Exception as e:
            print('[모델 스킵]', name, e)
    return fitted


def predict_all_candidates(fit_df, pred_df, feature_cols):
    cand = pd.DataFrame({'week': pred_df['week'].values})
    simple = simple_ma_candidates(pred_df).reset_index(drop=True)
    cand = pd.concat([cand, simple], axis=1)
    models = fit_ml_models(fit_df, feature_cols)
    Xp = pred_df[feature_cols]
    for name, model in models.items():
        cand[name] = np.maximum(model.predict(Xp), 0)
    return cand, models


def choose_candidate_by_cal(fit, cal, feature_cols):
    cal_cand, models = predict_all_candidates(fit, cal, feature_cols)
    rows = []
    params = {}
    for col in [c for c in cal_cand.columns if c != 'week']:
        raw = np.asarray(cal_cand[col], dtype=float)
        s, b = best_scale_offset(cal['firm_quantity'], raw, allow_offset=True)
        p = pack_round(np.maximum(raw*s + b, 0))
        row = eval_row(col, cal['firm_quantity'], p, fit['firm_quantity'])
        row['Scale'] = s
        row['Offset'] = b
        row['SelectScore'] = score_for_select(cal['firm_quantity'], p, fit['firm_quantity'])
        rows.append(row)
        params[col] = (s, b)
    table = pd.DataFrame(rows).sort_values(['RMSE_PASS','WAPE_PASS','SelectScore','RMSE','WAPE'], ascending=[False,False,True,True,True]).reset_index(drop=True)
    best = table.iloc[0]['Model']
    return best, params[best], table, models


def run():
    ensure_files_exist()
    weekly, raw, firm_col, confirmed_weeks = load_demand_weekly(DEMAND_PATH)
    weekly_ext, ext_cols = load_external_weekly(EXTERNAL_PATH)
    base, future_weeks = make_base(weekly, weekly_ext)
    feat = engineer_features(base, ext_cols)

    # FIRM 확정된 주차만 supervised target으로 사용
    confirmed = feat[feat['week'].isin(confirmed_weeks)].copy()
    required = ['firm_lag1','firm_lag2','firm_lag3','firm_lag4','firm_ma4','firm_ma8','forecast_quantity']
    usable = confirmed.dropna(subset=required).copy().reset_index(drop=True)
    if len(usable) < 20:
        raise ValueError(f'학습 가능한 FIRM 확정 주차가 너무 적습니다: {len(usable)}')

    test_n = min(HOLDOUT_CONFIRMED_WEEKS, max(4, len(usable)//5))
    train_eval = usable.iloc[:-test_n].copy()
    test = usable.iloc[-test_n:].copy()
    cal_n = min(INNER_CAL_CONFIRMED_WEEKS, max(3, len(train_eval)//5))
    fit = train_eval.iloc[:-cal_n].copy()
    cal = train_eval.iloc[-cal_n:].copy()

    exclude = {'week','firm_quantity'}
    feature_cols = [c for c in usable.columns if c not in exclude and pd.api.types.is_numeric_dtype(usable[c])]

    # calibration 기준 후보 선택
    selected, (scale, offset), cal_table, _ = choose_candidate_by_cal(fit, cal, feature_cols)

    # train_eval 전체로 재학습하고 holdout 평가
    test_cand, final_models = predict_all_candidates(train_eval, test, feature_cols)
    holdout_rows = []
    pred_cols = [c for c in test_cand.columns if c != 'week']
    applied_preds = {}
    for col in pred_cols:
        # 선택된 모델은 cal에서 고른 scale/offset 사용, 나머지는 각자 train_eval 마지막 cal로 다시 찾지 않고 원값 평가
        if col == selected:
            s, b = scale, offset
        else:
            s, b = 1.0, 0.0
        p = pack_round(np.maximum(np.asarray(test_cand[col]) * s + b, 0))
        applied_preds[col] = p
        row = eval_row(col, test['firm_quantity'], p, train_eval['firm_quantity'])
        row['Scale'] = s
        row['Offset'] = b
        row['Selected'] = (col == selected)
        holdout_rows.append(row)

    holdout_table = pd.DataFrame(holdout_rows).sort_values(['Selected','RMSE','WAPE'], ascending=[False,True,True]).reset_index(drop=True)
    selected_pred = applied_preds[selected]

    holdout_detail = test[['week','firm_quantity','forecast_quantity']].copy()
    holdout_detail['selected_pred'] = selected_pred
    holdout_detail['abs_error'] = np.abs(holdout_detail['firm_quantity'] - holdout_detail['selected_pred'])
    holdout_detail['selected_model'] = selected
    holdout_detail['scale'] = scale
    holdout_detail['offset'] = offset

    # 최종 모델은 usable 전체로 재학습해서 미래 4주 예측
    future_feat = feat[feat['week'].isin(future_weeks)].copy()
    future_cand, final_models2 = predict_all_candidates(usable, future_feat, feature_cols)
    if selected in future_cand.columns:
        future_raw = np.asarray(future_cand[selected], dtype=float)
    else:
        # 혹시 모델명이 누락되면 fixed_ma fallback
        future_raw = np.asarray(future_cand['fixed_ma'], dtype=float)
        selected = 'fixed_ma_fallback'
    future_pred = pack_round(np.maximum(future_raw*scale + offset, 0))
    forecast_df = pd.DataFrame({
        'week': future_weeks,
        'forecast_firm_quantity': future_pred,
        'raw_before_calibration': future_raw,
        'selected_model': selected,
        'scale': scale,
        'offset': offset,
        'note': 'FIRM 확정 기준 split + forecast_quantity 기반 후보 포함 + lag 1~16'
    })

    # 출력
    print('\n' + '='*115)
    print(f'[{TARGET_PN}] FIRM 확정 기준 v7 forecast-fix')
    print('='*115)
    print(f'FIRM 원본 행 수: {len(raw[raw[firm_col].astype(str).str.upper().str.strip().eq(FIRM_VALUE)]):,}')
    print(f'FORECAST 원본 행 수: {len(raw[raw[firm_col].astype(str).str.upper().str.strip().eq(FORECAST_VALUE)]):,}')
    print(f'FIRM 확정 주 범위: {confirmed_weeks.min():%Y-%m-%d} ~ {confirmed_weeks.max():%Y-%m-%d}')
    print(f'예측 대상 4주: {future_weeks[0]:%Y-%m-%d} ~ {future_weeks[-1]:%Y-%m-%d}')
    print(f'외부변수 lag 범위: 1~{MAX_EXTERNAL_LAG}주')
    print('\n[Split 정보]')
    print(f'usable_n={len(usable)}, fit_n={len(fit)}, cal_n={len(cal)}, test_n={len(test)}')
    print(f'fit_range: {fit.week.min():%Y-%m-%d} ~ {fit.week.max():%Y-%m-%d}')
    print(f'cal_range: {cal.week.min():%Y-%m-%d} ~ {cal.week.max():%Y-%m-%d}')
    print(f'test_range: {test.week.min():%Y-%m-%d} ~ {test.week.max():%Y-%m-%d}')

    print('\n[Calibration 후보 선택 결과 TOP 10]')
    print(cal_table[['Model','RMSE','WAPE','MAE','Sum Gap Rate','Scale','Offset','SelectScore']].head(10).to_string(index=False))
    print('\n[Holdout 성능 비교 TOP 15]')
    print(holdout_table[['Selected','Model','RMSE_PASS','WAPE_PASS','RMSE','WAPE','MAE','Sum Gap Rate','Actual Sum','Pred Sum','Scale','Offset']].head(15).to_string(index=False))
    print('\n[Holdout 상세]')
    print(holdout_detail.to_string(index=False))
    print('\n[미래 4주 예측]')
    print(forecast_df.to_string(index=False))

    # 그래프
    plt.figure(figsize=(14,6))
    hist = weekly[weekly['firm_quantity']>0].copy()
    plt.plot(hist['week'], hist['firm_quantity'], marker='o', label='Confirmed FIRM actual')
    plt.plot(holdout_detail['week'], holdout_detail['selected_pred'], marker='o', linestyle='-.', linewidth=3, label='Holdout selected prediction')
    plt.plot(forecast_df['week'], forecast_df['forecast_firm_quantity'], marker='*', markersize=14, linewidth=3, label='4-week forecast')
    plt.title(f'{TARGET_PN} FIRM Confirmed Forecast v7')
    plt.xlabel('Week')
    plt.ylabel('firm_quantity')
    plt.grid(True, alpha=.3)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    graph_path = f'{OUTPUT_PREFIX}_graph.png'
    plt.savefig(graph_path, dpi=200)
    plt.show()

    excel_path = f'{OUTPUT_PREFIX}_results.xlsx'
    split_info = pd.DataFrame({
        'item':['usable_n','fit_n','cal_n','test_n','fit_range','cal_range','test_range','selected_model','scale','offset','strict_no_test_leak'],
        'value':[len(usable),len(fit),len(cal),len(test),f'{fit.week.min():%Y-%m-%d}~{fit.week.max():%Y-%m-%d}',f'{cal.week.min():%Y-%m-%d}~{cal.week.max():%Y-%m-%d}',f'{test.week.min():%Y-%m-%d}~{test.week.max():%Y-%m-%d}',selected,scale,offset,STRICT_NO_TEST_LEAK]
    })
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        split_info.to_excel(writer, sheet_name='split_info', index=False)
        cal_table.to_excel(writer, sheet_name='calibration_eval', index=False)
        holdout_table.to_excel(writer, sheet_name='holdout_eval', index=False)
        holdout_detail.to_excel(writer, sheet_name='holdout_detail', index=False)
        forecast_df.to_excel(writer, sheet_name='forecast_4weeks', index=False)
        weekly.to_excel(writer, sheet_name='weekly_firm_forecast', index=False)
        usable[['week','firm_quantity','forecast_quantity'] + feature_cols[:80]].to_excel(writer, sheet_name='model_dataset_sample', index=False)

    print('\n[저장 완료]')
    print(excel_path)
    print(graph_path)
    if IN_COLAB:
        try:
            files.download(excel_path)
            files.download(graph_path)
        except Exception:
            pass

    return {
        'split_info': split_info,
        'calibration_eval': cal_table,
        'holdout_eval': holdout_table,
        'holdout_detail': holdout_detail,
        'forecast_4weeks': forecast_df,
        'selected_model': selected,
        'scale': scale,
        'offset': offset,
        'excel_path': excel_path,
        'graph_path': graph_path,
    }


result_v7 = run()
